In [ ]:
from board import Loc, Target, Cell, Node, Board, Transformer, parse, DIGITS, POS9

In [ ]:
from collections.abc import Generator


def iter_layer(board: Board, digit: int) -> Generator[Target]:
    for node in board:
        if digit in node.cell:
            yield Target(node.loc, digit)

## A puzzle


In [ ]:
puzzle = parse("""
.8.....52
.......87
....98...
4...3.6..
.2.7.....
.........
6..8.2...
...5.91..
9........
""")

## UI


In [ ]:
from typing import Iterable, Any
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from canvas import SudokuCanvas


In [ ]:
debug_view = w.Output()

canvas = SudokuCanvas()
canvas.draw_grid()
canvas[2].global_alpha = 0.5


def redraw_board():
    canvas.clear_highlights()
    canvas.draw_board(puzzle)


selecting_layers = w.SelectMultiple(description="Layers", options=(0,) + DIGITS, value=[0], indent=False)
selecting_layers.layout = w.Layout(width="auto", flex_flow="column", align_items="stretch")
selecting_layers.style.description_width = "auto"


@debug_view.capture()
def on_select_layer(change):
    selected = change.new
    canvas.clear_highlights()
    if selected != 0:
        with hold_canvas():
            for digit in selected:
                for target in iter_layer(puzzle, digit):
                    print("highlighting", target)
                    canvas.highlight_target(target)


selecting_layers.observe(on_select_layer, "value")


selecting_targets = w.SelectMultiple(description="Targets", options=[], value=[], indent=False)
selecting_targets.layout = w.Layout(flex_flow="column", align_items="stretch")
selecting_targets.style.description_width = "auto"


def on_select_target(change):
    selected = change.new
    canvas.clear_highlights()
    with hold_canvas():
        for target in selected:
            canvas.highlight_target(target)


selecting_targets.observe(on_select_target, "value")


selecting_links = w.SelectMultiple(description="Links", options=[], value=[], indent=False)
selecting_links.layout = w.Layout(flex_flow="column", align_items="stretch")
selecting_links.style.description_width = "auto"


def on_select_link(change):
    selected = change.new
    canvas.clear_highlights()
    with hold_canvas():
        for lnk in selected:
            canvas.highlight_link(lnk, style=lnk.kind)
        for lnk in selected:
            canvas.highlight_target(lnk[0])
            canvas.highlight_target(lnk[1])


selecting_links.observe(on_select_link, "value")


def load_options(widget, objects: Iterable[Any]):
    widget.options = [(str(obj), obj) for obj in objects]


w.HBox(
    [
        canvas,
        selecting_layers,
        selecting_targets,
        selecting_links,
    ],
)

In [ ]:
redraw_board()

In [ ]:
load_options(selecting_targets, anchors)
load_options(selecting_links, hardlinks | softlinks)

## Solving

kinda


In [ ]:
from typing import Iterable
from itertools import chain, combinations

In [ ]:
def fillempty(node: Node):
    if node.cell.is_empty:
        return Node(node.loc, Cell(DIGITS))
    else:
        return node

In [ ]:
def cleanup(board: Board) -> Transformer:

    def finals(locs):
        cells = [n.cell for n in board.slice(locs)]
        return set(c.final for c in cells if c.is_final)

    def cleanup(node: Node) -> Node:
        if node.cell.is_final:
            return node
        blkfinals = finals(node.loc.allblk())
        rowfinals = finals(node.loc.allrow())
        colfinals = finals(node.loc.allcol())
        allfinals = blkfinals | rowfinals | colfinals
        return Node(node.loc, Cell(set(node.cell) - allfinals))

    return cleanup


In [ ]:
puzzle = Board.transform(puzzle, fillempty)
puzzle = Board.transform(puzzle, cleanup(puzzle))

In [ ]:
class Link(tuple[Target, Target]):
    kind: str = ""

    def __str__(self):
        if self.kind == "HARD":
            return f"{self[0]}⟺{self[1]}"
        elif self.kind == "SOFT":
            return f"{self[0]}⟷{self[1]}"
        else:
            return f"{self[0]} ~ {self[1]}"

### hard links

Represent XOR relation

Criteria (for signular targets):

- only 2 drafts of same digit in a locality
- only 2 drafts in a cell


In [ ]:
def find_hardlinks_cell(node: Node):
    if len(node.cell) == 2:
        d1, d2 = node.cell
        yield Link((
            Target(node.loc, d1),
            Target(node.loc, d2),
        ))


def find_hardlinks_locality(nodes: Iterable[Node]):
    for d in DIGITS:
        sublayer = tuple(n for n in nodes if d in n.cell)
        if len(sublayer) == 2:
            n1, n2 = sublayer
            yield Link((
                Target(n1.loc, d),
                Target(n2.loc, d),
            ))


def find_hardlinks(board: Board) -> Generator[Link]:
    for node in board:
        yield from find_hardlinks_cell(node)
    for i in POS9:
        yield from find_hardlinks_locality(board.slice(Loc.forblk(i)))
        yield from find_hardlinks_locality(board.slice(Loc.forrow(i)))
        yield from find_hardlinks_locality(board.slice(Loc.forcol(i)))


In [ ]:
hardlinks = set(find_hardlinks(puzzle))
anchors = set(chain.from_iterable(hardlinks))

In [ ]:
for lnk in hardlinks:
    lnk.kind = "HARD"
    print(lnk)

### soft links

Represent NAND relation

Criteria:

- any 2 drafts of same digit in a locality
- any 2 drafts in a cell


In [ ]:
def check_softlink(t1: Target, t2: Target):
    l1 = t1.loc
    l2 = t2.loc
    if t1.seg == t1.seg:
        return l1.row == l2.row or l1.col == l2.col or l1.blk == l2.blk
    else:
        return l1 == l2


def find_softlinks(targets: Iterable[Target]) -> Generator[Link]:
    for t1, t2 in combinations(targets, r=2):
        if check_softlink(t1, t2):
            yield Link((t1, t2))

In [ ]:
# only checking already hardlinked anchors
# excuding dublicates

softlinks = set(find_softlinks(anchors)) - hardlinks

In [ ]:
for lnk in softlinks:
    lnk.kind = "SOFT"
    print(lnk)